## Genaration Policies

**ROOM → SCENARIO**

In [17]:
ROOM_SCENARIOS = {

    "Study Room": [
        "Study_Study Room",
        "Shutdown_Study Room",
        "Cleaning_Study Room",
        "Cleaning_stop_Study Room"
    ]
    
    # Other Scenarios continue here ...
}



**SCENARIO → COMMANDS**

In [18]:
SCENARIO_COMMANDS = defaultdict(list)

# ================= STUDY =================
SCENARIO_COMMANDS["Study_Study Room"] = [

    "I want to study now"
    
    # Commands under this scenario continue here
   
]

# ================= READING =================
SCENARIO_COMMANDS["Reading_Bedroom"] = [

    # Commands under this scenario
    
]

 # Other scenarios and commands continue here
    

**DIRECT COMMANDS**

In [19]:
DIRECT_DEVICE_COMMANDS = {

    # =====================================================
    # LIGHT
    # =====================================================
    "Light": {
        "On": [
            "Turn on the light"

        # Other varities of commands continue here
            
        ],

        "Off": [
            "Turn off the light"
            
            # Other varities of commands continue here

        ]
    },
        # Other devices and commands continue here
    
}


**SCENARIO → BASE DEVICE POLICY**

In [20]:
SCENARIO_BASE_DEVICES = {

    # ================= STUDY =================
    "Study_Study Room": [
        "Light", "AC", "Curtains",
        ("Smart Plug", "Radio"),
        ("Smart Plug", "Desk Lamp"),
        ("Smart Plug", "Air Purifier"),
        ("Smart Plug", "Computer")
    ],
    
    # Other scenarios and devices continue here
}


**TIME DISTRIBUTION**

In [21]:

TIME_DIST = {
    "Morning": 0.25,
    "Noon": 0.15,
    "Afternoon": 0.25,
    "Evening": 0.20,
    "Night": 0.15
}



**ENVIRONMENTAL HANDLING**

In [22]:
def sample_environment(time, room, scenario):

    # =========================================================
    # 1. OCCUPANCY 
    # =========================================================
    if "Sleep" in scenario:
        occupancy = random.choice([1, 2])

    # Other scenarios continue here

    elif "Shutdown" in scenario:
        occupancy = random.choice([1, 2])  

    else:
        occupancy = random.choice([1, 2]) 

    # =========================================================
    # 2. TEMPERATURE 
    # =========================================================
    if time == "Night":
        base_temp = random.randint(10, 25)
    
        # Other time period continue here
    
    else:
        base_temp = random.randint(15, 30)

    # room adjustment
    if room == "Kitchen":
        temp = base_temp + random.randint(1, 2)
    elif room == "Bedroom":
        temp = base_temp - random.randint(0, 2)
    else:
        temp = base_temp

    temp = max(10, min(temp, 35))

    # =========================================================
    # 3. LIGHT 
    # =========================================================

    # Time-based ambient light
    if time == "Night":
        time_base = random.randint(5, 40)
        
    # Other time period continue here

    else:
        time_base = random.randint(40, 80)

    # Room influence
    if room == "Kitchen":
        room_adjust = 10

    # Other rooms influence continue here

    elif room == "Dining Room":
        room_adjust = 5

    else:
        room_adjust = -5

    base = time_base + room_adjust

    light = max(5, min(base, 100))


    # =========================================================
    # NOISE
    # =========================================================
    if "Sleep" in scenario or "Shutdown" in scenario:
        noise = "Low"
    
    elif time == "Night":
        noise = random.choices(
            ["Low", "Medium"],
            [0.9, 0.1]
        )[0]
        
    # Other time period ccases continue here 
    
    else:
        noise = random.choice(["Low", "Medium", "High"])

    return temp, light, noise, occupancy



**ROOM DEVICES**

In [23]:
ROOM_DEVICES = {

    "Bedroom": [
        "SmartPlug_Noise Machine",
        "SmartPlug_Bed Lamp",
        "Curtains",
        "AC",
        "SmartPlug_Air Purifier",
        "SmartPlug_Radio",
        "SmartPlug_TV",
        "Light",
        "Mop Robot",
        "Robot Vacuum"
    ],
    
    # Other rooms continues here
}

**SCENARIO_DEVICE_POLICY**

In [24]:
SCENARIO_DEVICE_POLICY = {

    # ================= BEDROOM =================
    "Sleep Preparation_Bedroom": {
        "AC": "Adjust",
        "Air Purifier": "On",
        "Curtains": "Close",
        "Light": "Off",
        "Radio": "Off",
        "Bed Lamp": "On",
        "TV": "Off",
        "Noise Machine": "On"
    },

    # Other scenario policy continue here
}


**POLICY ENGINE**

In [25]:
def apply_device_policy(device, scenario, room, occupancy, time, temp, light, noise):

    # =========================================================
    # 1. SCENARIO POLICY (PRIMARY CONTROL)
    # =========================================================
    if scenario in SCENARIO_DEVICE_POLICY:

        policy = SCENARIO_DEVICE_POLICY[scenario]

        if device in policy:

            action = policy[device]

            # =========================================================
            # AC (VARIATION + TEMP AWARE)
            # =========================================================

            if device == "AC":

                # =====================================================
                # HARD SAFETY / PHYSICS CONSTRAINT
                # =====================================================
                if temp < 18:
                    return "Off", f"Temp={temp}°C"
        
                if action == "Off":
                    return "Off", f"Temp={temp}°C"
            
                # =====================================================
                # TIME OF DAY (DATASET-ALIGNED)
                # =====================================================
            
                if time == "Morning":
                    base_low, base_high = 21, 24
            
                # Other time perios contine here
            
                else:
                    # safety fallback (if unexpected label appears)
                    base_low, base_high = 21, 24
                

                target = random.randint(base_low, base_high)
            
                # natural variation (real-world noise)
                target += random.choice([-1, 0, 1])
            
                # safety bounds
                target = max(21, min(target, 26))
            
                return "Adjust", f"Temp={target}°C"
                
            # =========================================================
            # LIGHT 
            # =========================================================
            elif device == "Light":
            
                if action == "On":
                    return "On", "None"
            
                if action == "Off":
                    return "Off", "None"
            
                return action, "None"
                
            # =========================================================
            # CURTAINS (CONTEXT + COMMAND POLICY)
            # =========================================================
            elif device == "Curtains":
            
            
                # -----------------------------------------------------
                #  CONTEXT-AWARE DEFAULT BEHAVIOUR
                # -----------------------------------------------------
            
                if time in ["Morning", "Noon", "Afternoon"]:
                    return "Open", "None"
            
                elif time == "Evening":
                    return "Close", "None"
            
                elif time == "Night":
                    return "Close", "None"
            
                # -----------------------------------------------------
                # fallback
                # -----------------------------------------------------
                return action, "None"
            
            elif device in ["Radio", "Air Purifier", "TV", "Coffee Maker", "Mixer",
                "Toaster", "Electric Kettle", "Kitchen Light", "Desk Lamp", "Computer",
                "Noise Machine", "Bed Lamp", "Water Dispenser"]:

                if action == "On":
                    return "On", "None"

            elif device in ["Robot Vacuum", "Mop Robot", "Exhaust / Vent"]:

                if action == "On":
                    return "On", "None"
            
                return "Off", "None"

        return "Off", "None"


**DIRECT CONTROL POLICY**

In [26]:
import random

def apply_direct_device_policy(device, action, room, occupancy, time, temp, light, noise):

    # =========================================================
    # AC (DIRECT CONTROL — SAME LOGIC AS SCENE)
    # =========================================================
    if device == "AC":

        # =====================================================
        # HARD SAFETY / PHYSICS CONSTRAINT
        # =====================================================
        if temp < 18:
            return "Off", f"Temp={temp}°C"

        if action == "Off":
            return "Off", f"Temp={temp}°C"

        # =====================================================
        # TIME OF DAY
        # =====================================================
        if time == "Morning":
            base_low, base_high = 21, 24

            # Other time period continue here

        else:
            base_low, base_high = 21, 24

        # =====================================================
        # ROOM EFFECTS
        # =====================================================
        if room == "Bedroom":
            base_low -= 1
            base_high -= 1

            # Other room effect appear here

        # =====================================================
        # OCCUPANCY EFFECT
        # =====================================================
        if occupancy == 1:
            offset = 0
        elif occupancy == 2:
            offset = -1
        else:
            offset = -2

        # =====================================================
        # FINAL TEMPERATURE DECISION
        # =====================================================
        target = random.randint(base_low + offset, base_high + offset)
        target += random.choice([-1, 0, 1])
        target = max(18, min(target, 28))

        return "Adjust", f"Temp={target}°C"

    # =========================================================
    # LIGHT (DIRECT CONTROL)
    # =========================================================
    elif device == "Light":

        if action == "On":
            return "On", "None"

        if action == "Off":
            return "Off", "None"

        return action, "None"

    # =========================================================
    # CURTAINS (DIRECT CONTROL)
    # =========================================================
    elif device == "Curtains":

        if action == "Open":
            return "Open", "None"

        if action == "Close":
            return "Close", "None"

        # fallback contextual behaviour
        if time in ["Morning", "Noon", "Afternoon"]:
            return "Open", "None"
        else:
            return "Close", "None"

    # =========================================================
    # TV (DIRECT CONTROL)
    # =========================================================
    elif device == "TV":

        return action, "None"

    # =========================================================
    # SIMPLE ON/OFF DEVICES
    # =========================================================
    elif device in [
        "Radio", "Air Purifier", "Coffee Maker", "Mixer",
        "Toaster", "Electric Kettle", "Kitchen Light",
        "Desk Lamp", "Computer", "Noise Machine",
        "Bed Lamp", "Water Dispenser"
    ]:
        return action, "None"

    # =========================================================
    # ROBOTS / SYSTEM DEVICES
    # =========================================================
    elif device in ["Robot Vacuum", "Mop Robot", "Exhaust / Vent"]:

        if not action:
            return "Off", "None"

        return action, "None"

    # =========================================================
    # DEFAULT FALLBACK
    # =========================================================
    return action, "None"

**GENERATION**

In [27]:

def generate_device_set(scenario, room, occupancy, time, temp, light, noise):

    base_devices = SCENARIO_BASE_DEVICES.get(scenario, [])
    results = []

    for device in base_devices:

        # =========================
        # SMART PLUG CASE
        # =========================
        if isinstance(device, tuple):
    
            base_device = device

            action, params = apply_device_policy(
                base_device, scenario, room, occupancy, time, temp, light, noise
            )

            results.append({
                "Device": base_device,
                "Action": action,
                "Parameter": params if params else None
            })

    return results



**PREVIOUS ACTION GENERATOR HELPER**

In [28]:
def infer_previous_state(device, final_action):
    
    if device == "AC":
        return random.choice(["Off", "Adjust"])

    # Other device appear here

    # default fallback
    return random.choice(["On", "Off"])

**Balancing dataset**

In [29]:

# =========================================================
# BALANCED DATASET GENERATOR
# =========================================================

def generate_dataset(samples_per_combination=6):

    data = []
    stats = defaultdict(int)

    # =====================================================
    # 1. SCENE-LEVEL COMMAND GENERATION (NO FILTERING)
    # =====================================================

    for room in ROOM_SCENARIOS:

        for scenario in ROOM_SCENARIOS[room]:

            commands = SCENARIO_COMMANDS.get(
                scenario,
                ["Default command"]
            )

            for command in commands:

                for time in TIME_DIST.keys():

                    for _ in range(samples_per_combination):

                        event_id = str(uuid.uuid4())

                        # -------------------------------
                        # ENVIRONMENT
                        # -------------------------------
                        temp, light, noise, occupancy = sample_environment(
                            time,
                            room,
                            scenario
                        )

                        # -------------------------------
                        # DEVICE ACTIONS (NO ROOM FILTER HERE)
                        # -------------------------------
                        device_set = generate_device_set(
                            scenario=scenario,
                            room=room,
                            occupancy=occupancy,
                            time=time,
                            temp=temp,
                            light=light,
                            noise=noise
                        )

                        if not device_set:
                            continue

                        for item in device_set:

                            device = item["Device"]
                            action = item["Action"]

                            previous_state = infer_previous_state(device, action)

                            data.append({
                                "Event_ID": event_id,
                                "Intent_Type": "Scene_Control",
                                "Command": command,
                                "Device": device,
                                "Current State": previous_state,
                                "Time": time,
                                "Light": f"{light}%",
                                "Temperature": f"{temp}°C",
                                "Noise": noise,
                                "Occupancy": occupancy,
                                "Location": room,
                                "Scenario": scenario,
                                "Action": action,
                                "Parameters": item["Parameter"]
                                if item["Parameter"] else "None"
                            })

                            stats[f"Room::{room}"] += 1
                            stats[f"Scenario::{scenario}"] += 1
                            stats[f"Command::{command}"] += 1
                            stats[f"Time::{time}"] += 1
                            stats[f"Device::{device}"] += 1
                            stats[f"Action::{action}"] += 1

    # =====================================================
    # 2. DIRECT DEVICE CONTROL (WITH ROOM FILTER)
    # =====================================================

    for room in ROOM_DEVICES:

        for device in DIRECT_DEVICE_COMMANDS:

            # ONLY direct control uses room constraint
            if device not in ROOM_DEVICES.get(room, []):
                continue

            for action in DIRECT_DEVICE_COMMANDS[device]:

                commands = DIRECT_DEVICE_COMMANDS[device][action]

                for command in commands:

                    for time in TIME_DIST.keys():

                        for _ in range(samples_per_combination):

                            event_id = str(uuid.uuid4())

                            temp, light, noise, occupancy = sample_environment(
                                time,
                                room,
                                "Direct_Control"
                            )

                            previous_state = infer_previous_state(device, action)

                            final_action, parameter = apply_direct_device_policy(
                                device=device,
                                action=action,
                                room=room,
                                occupancy=occupancy,
                                time=time,
                                temp=temp,
                                light=light,
                                noise=noise
                            )

                            data.append({
                                "Event_ID": event_id,
                                "Intent_Type": "Direct_Device_Control",
                                "Command": command,
                                "Device": device,
                                "Current State": previous_state,
                                "Time": time,
                                "Light": f"{light}%",
                                "Temperature": f"{temp}°C",
                                "Noise": noise,
                                "Occupancy": occupancy,
                                "Location": room,
                                "Scenario": "Direct_Control",
                                "Action": final_action,
                                "Parameters": parameter
                            })

                            stats[f"Room::{room}"] += 1
                            stats[f"Scenario::Direct_Control"] += 1
                            stats[f"Command::{command}"] += 1
                            stats[f"Time::{time}"] += 1
                            stats[f"Device::{device}"] += 1
                            stats[f"Action::{action}"] += 1

    # =====================================================
    # FINAL DATAFRAME
    # =====================================================

    df = pd.DataFrame(data)
    df = df.sample(frac=1).reset_index(drop=True)

    return df

**Generating Dataset**

In [ ]:
# =========================================================
# RUN
# =========================================================
df = generate_dataset(samples_per_combination=8)



# ensure directory exists
os.makedirs("./", exist_ok=True)

df.to_csv(
    "./SmartHomeIoTNLU.csv",
    index=False,
    encoding="utf-8-sig"
)
